# Caching & Persistence — Speeding up Reuse

When and how to use cache() and persist() in Spark

By now, you’ve seen how Spark builds DAGs lazily and runs tasks only when an action is triggered. This design is elegant — it ensures Spark optimizes the whole plan before touching the data. But there’s a downside: every time you call an action, Spark recomputes the entire chain of transformations from the beginning.

That’s fine if your job is short. But what if you need to reuse the same transformed dataset multiple times — say, one action writes it out to Parquet, another computes aggregates, and a third feeds into a model? Without help, Spark will start from scratch each time, wasting effort.

That’s where caching and persistence come in. They are Spark’s way of saying: “I’ll keep this DataFrame handy in memory (or disk) so you don’t have to redo the work.”

### Recomputing the Same Work
Let’s start with a simple example:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("day17-cache").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_csv.csv")

df_transformed = df.withColumn("rev_with_tax", col("revenue") * 1.05)

In [0]:
print(df_transformed.count())  
df_transformed.write.mode("overwrite").parquet("/Volumes/thedataengineering_00/data/sampledata/sales_caching")

Spark will compute the transformations twice — once for the count, once for the write. It doesn’t remember the result after the first action.

This can become painfully expensive if your transformations involve wide shuffles or heavy aggregations.

### Enter cache()
The simplest way to avoid recomputation is to tell Spark: “Please keep this DataFrame in memory.”

In [0]:
df_cached = df_transformed.cache()
print(df_cached.count())   # First action triggers computation + caches result
print(df_cached.count())   # Second action reuses cached data

On the first action, Spark computes and stores partitions in memory on executors. On subsequent actions, it pulls them straight from cache.

For iterative workflows (like machine learning, where you scan the same data again and again), caching can save huge amounts of time.

### persist() — More Control Than Cache
While cache() is just a shorthand for persist(StorageLevel.MEMORY_AND_DISK), persist() lets you control where Spark stores data:

Memory only: fastest, but risky if dataset is large — some partitions may be recomputed if they don’t fit.

Memory + Disk: safest default; if memory is full, spill to disk.

Disk only: useful if you don’t want to consume RAM.

Serialized vs deserialized: trade-off between CPU cost and space efficiency.

Example:

In [0]:
from pyspark import StorageLevel

df_persisted = df_transformed.persist(StorageLevel.MEMORY_AND_DISK)
df_persisted.count()   # triggers computation + stores partitions

If the dataset is too big for memory, Spark will keep as much as it can in RAM and write the rest to local disk on executors.

When Caching Helps
To appreciate caching, imagine this pipeline:

- Read raw sales data.
- Clean missing values.
- Join with customer dimension.
- Add derived metrics.
- Use results in three different actions:
            Count rows.
            Save as Parquet.
            Compute aggregates.

Without caching, Spark would recompute steps 1–4 three times. With caching, it does them once, keeps the result handy, and reuses it for all actions.

### Seeing Cache in Action
One of the best ways to understand cache is to use the Spark UI.

Run an action without caching → you’ll see all transformations recomputed each time.

Run the same with caching → you’ll notice the second action skips earlier stages and reuses cached partitions.

This is eye-opening for freshers, because it makes clear that caching isn’t about correctness — it’s about performance.

### The Risks of Overusing Cache
Like every optimization, caching can backfire if misused.

If you cache huge datasets that don’t fit in memory, Spark may spend more time spilling to disk than recomputing.

Cached data sticks around until you explicitly call unpersist(). If you forget, you may fill up executors’ memory and cause other jobs to slow down.

Good practice is:

Cache only if you’re going to reuse the dataset multiple times.

Drop (unpersist) once you’re done.

In [0]:
df_cached.unpersist()


### An Analogy

Think of Spark transformations like cooking a dish. Without caching, every time someone asks for a portion, you start chopping vegetables and cooking from scratch. With caching, you prepare a big batch once and keep it warm. Everyone gets their plate faster — but if you make too much and don’t have enough pots (memory), the kitchen gets messy.

# Wrapping Up
Today you learned how to stop Spark from repeating itself unnecessarily:

By default, Spark recomputes the whole DAG for each action.

cache() keeps results in memory for reuse.

persist() gives you control over storage levels (memory, disk, serialized).

Use caching when a dataset is reused across multiple actions, but avoid over-caching.

Caching is one of those tools that turns Spark from “it works” into “it works efficiently.” It’s the difference between waiting an hour for iterative analysis and finishing in minutes.

That’s Day 17. You now know how to use caching and persistence wisely.